# Chapter 6 &mdash; Pruning Unreachable States

**Concept 3 of the Chapter 6 decomposition:** *Pruning Unreachable States*

The product construction makes disconnected states; BFS from $q_0$ for $|Q|-1$ steps keeps only what matters.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Pruning-Unreachable/Concept-Pruning-Unreachable.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


The product construction creates **every** pair, including pairs no input can ever
reach. Those states are harmless but useless: they clutter the drawing and slow every
later algorithm.

**Pruning** is a breadth-first search from $q_0$. Since the longest simple path has
$|Q|-1$ edges, $|Q|-1$ rounds suffice &mdash; that bound is the reason the algorithm
terminates without a visited-set argument.

Pruning is **not** minimization: it removes states you cannot get to, not states you
cannot tell apart.

## 2. Definitions

### Two machines whose product has unreachable pairs

In [ ]:
A = md2mc('''DFA
I : 0 -> F
I : 1 -> I
F : 0 -> F
F : 1 -> I
''')
B = md2mc('''DFA
I : 0 -> I
I : 1 -> F
F : 0 -> I
F : 1 -> F
''')

### Reachability by BFS, bounded by $|Q|-1$ rounds

In [ ]:
def reachable(D):
    frontier, seen = {D["q0"]}, {D["q0"]}
    for _ in range(len(D["Q"]) - 1):
        nxt = {step_dfa(D, q, c) for q in frontier for c in D["Sigma"]} - seen
        if not nxt: break
        seen |= nxt; frontier = nxt
    return seen

<!-- nav-strip -->

---

&larr;&nbsp;[Ch6&nbsp;2.&nbsp;The Product Construction: Union and Intersection of DFA](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Product-Construction/Concept-Product-Construction.ipynb) &nbsp;&middot;&nbsp; [**Chapter 6** index](https://github.com/ganeshutah/Jove/blob/master/Chapter6/README.md) &nbsp;&middot;&nbsp; [Ch6&nbsp;4.&nbsp;Language Equivalence Checking by Lock-Step Search, with Counterexamples](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Language-Equivalence-Checking/Concept-Language-Equivalence-Checking.ipynb)&nbsp;&rarr;

---

## 3. Tests

The raw product has states the BFS never touches.

In [ ]:
P = union_dfa(A, B)
r = reachable(P)
print("product states : %d,  reachable : %d" % (len(P["Q"]), len(r)))
print("unreachable    :", sorted(P["Q"] - r))

`pruneUnreach` removes exactly those, and the language is unchanged.

In [ ]:
Pr = pruneUnreach(P)
print("after pruneUnreach : %d states" % len(Pr["Q"]))
assert Pr["Q"] == reachable(P)
assert langeq_dfa(P, Pr)
print("same language? ", langeq_dfa(P, Pr))

Pruning is **not** minimization &mdash; they remove different things.

In [ ]:
print("raw product   : %d states" % len(P["Q"]))
print("pruned        : %d states" % len(Pr["Q"]))
print("minimized     : %d states" % len(min_dfa(P)["Q"]))
print()
print("pruning removes UNREACHABLE states;")
print("minimizing then merges INDISTINGUISHABLE ones (Concepts 6-10).")

The $|Q|-1$ bound really is enough: a longer chain still converges in time.

In [ ]:
chain = md2mc('''DFA
I  : 0 -> A
A  : 0 -> B
B  : 0 -> C
C  : 0 -> F
F  : 0 -> F
I  : 1 -> I
A  : 1 -> A
B  : 1 -> B
C  : 1 -> C
F  : 1 -> F
''')
print("|Q| =", len(chain["Q"]), " all reachable?", reachable(chain) == chain["Q"])
assert reachable(chain) == chain["Q"]

## 4. Animation

The pruned union &mdash; only the pairs an input can actually reach.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(pruneUnreach(union_dfa(A, B)), FuseEdges=True)

## 5. Exercises


1. Construct a DFA where **half** the product states are unreachable.
2. Why does BFS need only $|Q|-1$ rounds and not $|Q|$?
3. Can an unreachable state be *distinguishable* from a reachable one? Does it matter?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter6/Concept-Pruning-Unreachable')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')